# Setup
Das Notebook dient dazu die Images zu downloaden.  
Wichtig hierfür sind in der Config unteranderem MAX_WORKER und die Downloadgeschwindigkeit liegt an:
- MAX_WORKERS COUNT
- Rechenleistung
- Internetverbindung
- Cloud Mangagment (emfohlen lokale Speicherung)

In [ ]:
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
import yaml
from PIL import Image
from requests.adapters import HTTPAdapter, Retry
from tqdm import tqdm

thread_local = threading.local()

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")

def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


cfg_file = find_upwards("config.yaml")
assert cfg_file, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(cfg_file.read_text())


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"
IMAGE_PATH = Path(CFG["img_download_path"]).expanduser()
IMG_SIZE = CFG["download_image_size"]
metadata = pd.read_parquet(DATA_PATH_META)
download_metadata = metadata[metadata["split"].isin(["train","database", "query"])].copy()

# Set Max Workers -> Recommendation for new Laptop 96
MAX_WORKERS = CFG["download_workers"]
if MAX_WORKERS == "auto":
    MAX_WORKERS = 96
else:
    MAX_WORKERS = int(MAX_WORKERS)


# Get Token
if env_file := find_upwards(".env"):
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

TOKEN = os.environ.get("MAPILLARY_TOKEN", "")
assert TOKEN.startswith("MLY|"), ("Kein Mapillary-Token. Datei .env anlegen:\n MAPILLARY_TOKEN=MLY|dein|token")


print("Download Location: ", IMAGE_PATH)
print(f"Download Threads: {MAX_WORKERS}")
print(f"Zu ladende Bilder: {len(download_metadata):,}")

# API Anfrage
  
Erstelle session, frage API an 

In [ ]:
# Erstelle Session über HTTP Adapter -> https oder http
def make_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=True,
        respect_retry_after_header=True,
    )

    adapter = HTTPAdapter(
        max_retries=retry, pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS
    )

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


def get_session():
    if not hasattr(thread_local, "session"):
        thread_local.session = make_session()

    return thread_local.session


def download_image(image_id):

    image_id = str(image_id)
    image_file = IMAGE_PATH / f"{image_id}.jpg"

    # Bereits vorhandenes, nicht-leeres Bild überspringen
    if image_file.exists() and image_file.stat().st_size > 0:
        return "exists"

    session = get_session()

    try:
        api_url = f"https://graph.mapillary.com/{image_id}"

        params = {
            "fields": f"id,thumb_{IMG_SIZE}_url",
            "access_token": TOKEN,
        }

        api_response = session.get(
            api_url,
            params=params,
            timeout=(10, 30),
        )

        api_response.raise_for_status()

        data = api_response.json()

        image_url = data.get(f"thumb_{IMG_SIZE}_url")

        if not image_url:
            print(f"Keine Bild-URL für {image_id}")
            return "failed"

        image_response = session.get(image_url, timeout=(10, 60))
        image_response.raise_for_status()

        if not image_response.content:
            print(f"Leere Bildantwort für {image_id}")
            return "failed"

        image_file.write_bytes(image_response.content)

        return "downloaded"

    except requests.RequestException as e:
        status = e.response.status_code if e.response is not None else "keine Antwort"

        print(f"Fehler bei {image_id}: {type(e).__name__}: {status}")

        return "failed"

    except (ValueError, KeyError) as e:
        print(f"Ungültige Mapillary-Antwort für {image_id}: {type(e).__name__}")

        return "failed"

    except Exception as e:
        # Fängt unerwartete Fehler ab, damit ein einzelnes Bild
        # nicht den gesamten 98k-Download stoppt.
        print(f"Unerwarteter Fehler bei {image_id}: {type(e).__name__}: {e}")

        return "failed"


# Download

In [ ]:

# Ein Verzeichnisdurchlauf statt 332.868 einzelner Abfragen im Threadpool:
# beim wiederholten Lauf ist praktisch alles schon da.
vorhandene_ids = {
    eintrag.name[:-4]
    for eintrag in os.scandir(IMAGE_PATH)
    if eintrag.name.endswith(".jpg") and eintrag.stat().st_size > 0
}

alle_ids = [str(i) for i in download_metadata["image_id"]]
fehlende_ids = [i for i in alle_ids if i not in vorhandene_ids]

print(f"Bereits vorhanden: {len(alle_ids) - len(fehlende_ids):,}")
print(f"Zu holen:          {len(fehlende_ids):,}")

results = {}

if fehlende_ids:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(download_image, image_id): image_id
            for image_id in fehlende_ids
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc="Bilder herunterladen"):
            image_id = futures[future]
            try:
                results[image_id] = future.result()
            except Exception as e:
                print(f"Unerwarteter Fehler bei {image_id}: {type(e).__name__}: {e}")
                results[image_id] = "failed"

# Nur diese muessen anschliessend auf Unversehrtheit geprueft werden.
neu_geladene_ids = [i for i, r in results.items() if r == "downloaded"]
failed_ids = [i for i, r in results.items() if r == "failed"]
image_files = list(IMAGE_PATH.glob("*.jpg"))
FAILED_IMAGE_PATH = PROCESSED_DIR / "failed_image_download.txt"
FAILED_IMAGE_PATH.write_text("\n".join(map(str, failed_ids)))


# Auswertung erster Download
print("=" * 50)
print("DOWNLOAD ERGEBNISSE")
print("=" * 50)
print(f"Neu geladen:                {len(neu_geladene_ids):,}")
print(f"Bereits vorhanden:          {len(alle_ids) - len(fehlende_ids):,}")
print(f"Fehlgeschlagene:            {list(results.values()).count('failed'):,}")
print("=" * 50)
print(f"Fehlgeschlagene Bilder:     {len(failed_ids):,}")
print(f"Liste gespeichert unter:    {FAILED_IMAGE_PATH}")
print(f"Lokale JPGs:                {len(image_files):,}")
print("=" * 50)


# Kaputte Bilder

In [ ]:
# Abgebrochene Downloads erkennen. Nur die Bilder aus diesem Lauf -- an den
# uebrigen kann sich nichts geaendert haben. PRUEFE_ALLE liest den gesamten
# Bestand, was bei 330k Dateien etliche Minuten dauert.

PRUEFE_ALLE = False

if PRUEFE_ALLE:
    zu_pruefen = list(IMAGE_PATH.glob("*.jpg"))
else:
    zu_pruefen = [IMAGE_PATH / f"{i}.jpg" for i in neu_geladene_ids]

bad = []

for img in tqdm(zu_pruefen, desc="Bilder pruefen"):
    try:
        with Image.open(img) as im:
            im.verify()
    except Exception:
        bad.append(img)

print(f"Geprueft: {len(zu_pruefen):,}   kaputt: {len(bad)}")

if bad:
    for img in bad:
        img.unlink()
    print(
        f"{len(bad)} geloescht -- die Download-Zelle erneut ausfuehren, "
        "dann werden sie neu geholt."
    )


In [ ]:
# ------------------------------------------------------------
# Aufklaerung: Wie gut sind Mapillarys Detections abgedeckt?
#
# Entscheidungsgrundlage dafuer, ob semantisches Re-Ranking lohnt.
# Laedt keine Bilder, nur eine kleine JSON-Antwort je Bild.
#
# Einmalige Erhebung -- die Abdeckung aendert sich nicht dadurch, dass die
# Pipeline erneut laeuft. Das Ergebnis wird gespeichert und beim naechsten
# Mal nur noch ausgegeben. NEU_MESSEN erzwingt die Messung.
# ------------------------------------------------------------

import collections
import json

N_PROBE = 500
NEU_MESSEN = False
PROBE_PATH = PROJECT_ROOT / "results" / "detections_probe.json"


def show_summary(befund):
    print("=" * 58)
    print("DETECTION-ABDECKUNG")
    print("=" * 58)
    print(f"Geprueft:                   {befund['n_geprueft']:,}")
    print(f"Bilder mit Detections:      {befund['anteil_mit_detections'] * 100:.1f} %")
    print(f"Detections je Bild:         Median {befund['median_detections']:.0f}")
    print(f"Verschiedene Klassen:       {befund['n_klassen']:,}")
    print()
    print("Haeufigste Klassen:")
    for k, v in list(befund["haeufigste"].items())[:15]:
        print(f"  {k:<45s} {v:>6,}")
    print("=" * 58)
    print("Faustregel: unter 50 % Abdeckung ODER Median < 3 Detections/Bild")
    print("-> Re-Ranking lohnt nicht, Befund als Absatz in die Fehleranalyse.")


if PROBE_PATH.exists() and not NEU_MESSEN:
    befund = json.loads(PROBE_PATH.read_text())
    print(f"Gespeicherter Befund vom {befund['datum']} "
          f"({PROBE_PATH.name}), NEU_MESSEN = True misst erneut.\n")
    show_summary(befund)

else:
    db_ids = metadata.loc[metadata["split"] == "database", "image_id"].sample(
        N_PROBE, random_state=42
    )

    session = make_session()
    n_with = 0
    n_detections = []
    klassen = collections.Counter()
    n_fehler = 0

    for image_id in tqdm(db_ids, desc="Detections pruefen"):
        try:
            r = session.get(
                f"https://graph.mapillary.com/{image_id}/detections",
                params={"access_token": TOKEN, "fields": "value"},
                timeout=30,
            )
            r.raise_for_status()
            d = r.json().get("data", [])
        except Exception:
            n_fehler += 1
            continue

        n_detections.append(len(d))
        if d:
            n_with += 1
            klassen.update(x["value"] for x in d)

    n_ok = len(n_detections)
    befund = {
        "datum": pd.Timestamp.now().strftime("%Y-%m-%d"),
        "n_geprueft": n_ok,
        "n_fehler": n_fehler,
        "anteil_mit_detections": n_with / max(n_ok, 1),
        "median_detections": float(pd.Series(n_detections).median()) if n_detections else 0.0,
        "mittel_detections": float(pd.Series(n_detections).mean()) if n_detections else 0.0,
        "n_klassen": len(klassen),
        "haeufigste": dict(klassen.most_common(30)),
    }

    PROBE_PATH.parent.mkdir(parents=True, exist_ok=True)
    PROBE_PATH.write_text(json.dumps(befund, indent=2))

    show_summary(befund)
    print(f"\nGespeichert unter {PROBE_PATH}")
